# 01 — Pipeline Quick Start: Phase Classification in 30 Seconds

This notebook demonstrates the complete GNN-HVA pipeline for quantum phase
classification on the Transverse-Field Ising Model (TFIM).

**What you'll see:**
- Phase 1: Exact ground truth computation
- Phase 2: VQE optimization with warm-start sweep
- Phase 3: MPNN training (GINConv predictor)
- Phase 4: Zero-shot deployment on unseen h-values

**Runtime:** ~30s on CPU (N=6, p=2, chain_1d)

**Requirements:** `pip install -e .` from the repository root.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Pipeline core imports
from qmbp_simulation import (
    HamiltonianBuilder, make_lattice, ClassicalSolver,
    HVACircuitBuilder, VQEOptimizer, VQEConfig,
    PipelineRunner,
)
from qmbp_simulation.models.model_registry import get_model_spec
from qmbp_simulation.predictors import build_graph_dataset, train_mpnn
from qmbp_simulation.execution import NoiselessBackend

print("✅ All imports successful")

## Configuration

We use the simplest viable configuration: N=6 qubits, p=2 HVA layers,
chain_1d topology, TFIM longitudinal model.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
N = 6               # Number of qubits
P = 2               # HVA layers
TOPOLOGY = "chain_1d"
MODEL = "tfim_longitudinal"
H_MIN = 1.0         # Minimum transverse field
H_MAX = 3.5         # Maximum transverse field
N_TRAIN = 20        # Training h-points (Phase 2)
N_TEST = 5          # Test h-points (Phase 4 deployment)
SEED = 42

# Get model specification (encapsulates Hamiltonian + circuit + defaults)
spec = get_model_spec(MODEL)
print(f"Model: {spec.name} — {spec.description}")
print(f"Parameters per layer: {spec.params_per_layer}")
print(f"Total params (p={P}): {spec.total_params_for_p(P)}")

## Visualize: Lattice Topology & Pipeline

Before running the pipeline, let's see what system we're working with.

In [ ]:
from viz_helpers import draw_lattice, draw_pipeline_diagram, draw_hva_structure, draw_circuit

# 1. Pipeline architecture overview
fig = draw_pipeline_diagram()
plt.show()

# 2. Lattice topology (spin sites + interactions)
lattice_viz = make_lattice(TOPOLOGY, N, J=1.0, h=2.0)
fig = draw_lattice(TOPOLOGY, N, lattice_viz.edges, h_value=2.0,
                   title=f"Spin System: {MODEL} on {TOPOLOGY} (N={N})")
plt.show()

# 3. HVA circuit structure schematic
fig = draw_hva_structure(N, P, lattice_viz.edges, model_name=MODEL.upper())
plt.show()

## Phase 1: Classical Ground Truth

Compute exact energies and spectral gaps via numpy diagonalization.

In [ ]:
# Generate h-grid (descending for warm-start sweep)
h_train = np.linspace(H_MAX, H_MIN, N_TRAIN)
h_test = np.array([1.2, 1.8, 2.3, 2.8, 3.2])  # Unseen test points

# Phase 1: Exact diagonalization
builder = HamiltonianBuilder()
solver = ClassicalSolver()

exact_results = []
for h in h_train:
    lattice_h = make_lattice(TOPOLOGY, N, J=1.0, h=float(h))
    H = spec.build_hamiltonian(lattice_h, **spec.hamiltonian_kwargs)
    result = solver.solve(H, lattice_h)
    exact_results.append(result)

e_exact = np.array([r.ground_energy for r in exact_results])
gaps = np.array([r.gap for r in exact_results])

print(f"✅ Phase 1 complete: {len(exact_results)} h-points")
print(f"   Energy range: [{e_exact.min():.4f}, {e_exact.max():.4f}]")
print(f"   Gap range: [{gaps.min():.4f}, {gaps.max():.4f}]")

## Phase 2: VQE Warm-Start Optimization

Descending sweep from h_max → h_min. Each point inherits θ* from the
previous point (warm-start), following Mele et al. (2022) and Puig et al. (2025).

In [ ]:
# Build the HVA circuit
lattice_ref = make_lattice(TOPOLOGY, N, J=1.0, h=float(h_train[0]))
circuit, theta_params = spec.create_circuit(N, P, lattice_ref, **spec.circuit_kwargs)
print(f"Circuit: {circuit.num_qubits} qubits, {circuit.num_parameters} parameters")

# Draw the quantum circuit
fig = draw_circuit(circuit, title=f"HVA Circuit — {MODEL} (N={N}, p={P})")
plt.show()
print(f"2-qubit gates: {circuit.num_nonlocal_gates()}, depth: {circuit.depth()}")

In [ ]:
# VQE optimizer with warm-start
vqe_config = VQEConfig(
    method="L-BFGS-B",
    maxiter=500,
    n_restarts=3,
    restart_sigma=0.1,
)
optimizer = VQEOptimizer(config=vqe_config, backend=NoiselessBackend(), seed=SEED)

# Descending sweep
vqe_results = optimizer.descending_sweep(
    h_values=h_train,
    circuit=circuit,
    lattice=lattice_ref,
    exact_data=exact_results,
)

theta_opt = np.array([r.theta_opt for r in vqe_results])
fidelities = np.array([r.fidelity for r in vqe_results])
de_gaps = np.array([abs(r.energy - e) / g for r, e, g in zip(vqe_results, e_exact, gaps)])

print(f"\n✅ Phase 2 complete: {len(vqe_results)} points optimized")
print(f"   Mean fidelity: {fidelities.mean():.4f}")
print(f"   Mean ΔE/gap: {de_gaps.mean():.4f} ({de_gaps.mean()*100:.2f}%)")
print(f"   All pass (<5%): {(de_gaps < 0.05).all()}")

## Phase 3: MPNN Training

Train a GINConv MPNN to predict θ* directly from the Hamiltonian graph.

In [ ]:
# Build PyTorch Geometric dataset
dataset = build_graph_dataset(
    lattice=lattice_ref,
    h_values=h_train,
    theta_opt=theta_opt,
    e_exact=e_exact,
    fidelities=fidelities,
    fidelity_threshold=0.0,  # No filtering for this demo
)
print(f"Dataset: {len(dataset)} graphs, {dataset[0].x.shape[1]} node features")

# Train MPNN (auto-creates model when first arg is None)
model, train_result = train_mpnn(
    None,  # auto-create MPNNPredictor from dataset shape
    dataset=dataset,
    hidden_dim=64,
    n_layers=3,
    n_epochs=3000,
    lr=1e-3,
    patience=200,
    seed=SEED,
)
print(f"\n✅ Phase 3 complete")
print(f"   MPNN: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"   Final MSE: {train_result['final_mse']:.6f}")
print(f"   Stopped early: {train_result['stopped_early']}")

In [ ]:
# Visualize what the MPNN sees as input
from viz_helpers import draw_gnn_input_graph
fig = draw_gnn_input_graph(N, lattice_ref.edges, h_value=2.0)
plt.show()

## Phase 4: Zero-Shot Deployment

Predict θ at unseen h-values and evaluate energy accuracy.

In [ ]:
import torch
from torch_geometric.data import Data

backend = NoiselessBackend()
deploy_results = []

model.eval()
for h in h_test:
    # Build graph for this h-point
    lattice_h = make_lattice(TOPOLOGY, N, J=1.0, h=float(h))
    H = spec.build_hamiltonian(lattice_h, **spec.hamiltonian_kwargs)
    gt = solver.solve(H, lattice_h)
    
    # Construct input graph (same as training)
    edge_index_np, coord = builder.build_graph_data(lattice_h)
    edge_index = torch.tensor(edge_index_np, dtype=torch.long)
    h_feat = np.full(N, float(h))
    x = torch.tensor(np.stack([h_feat, coord.astype(float)], axis=1), dtype=torch.float32)
    data = Data(x=x, edge_index=edge_index)
    data.batch = torch.zeros(N, dtype=torch.long)
    
    # Predict θ
    with torch.no_grad():
        theta_pred = model(data).numpy().flatten()
    
    # Evaluate energy
    e_pred = backend.evaluate(circuit, H, theta_pred)
    de_gap = abs(e_pred - gt.ground_energy) / gt.gap
    
    deploy_results.append({
        "h": float(h),
        "e_exact": gt.ground_energy,
        "e_predicted": e_pred,
        "de_gap": de_gap,
        "pass": de_gap < 0.05,
    })

# Summary
pass_rate = sum(1 for r in deploy_results if r["pass"]) / len(deploy_results)
mean_de_gap = np.mean([r["de_gap"] for r in deploy_results])

print(f"\n✅ Phase 4 — Deployment Results")
print(f"{'h':>6} {'E_exact':>10} {'E_pred':>10} {'ΔE/gap':>8} {'Pass':>5}")
print("-" * 45)
for r in deploy_results:
    print(f"{r['h']:6.2f} {r['e_exact']:10.4f} {r['e_predicted']:10.4f} "
          f"{r['de_gap']:8.4f} {'✅' if r['pass'] else '❌'}")
print(f"\nPassRate: {pass_rate*100:.0f}% | Mean ΔE/gap: {mean_de_gap*100:.2f}%")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Energies
ax = axes[0]
ax.plot(h_train, e_exact, 'k-', lw=2, label='Exact')
ax.plot(h_train, [r.energy for r in vqe_results], 'b.', ms=4, label='VQE')
ax.scatter([r['h'] for r in deploy_results],
           [r['e_predicted'] for r in deploy_results],
           c='red', s=80, zorder=5, marker='*', label='MPNN (zero-shot)')
ax.set_xlabel('h (transverse field)')
ax.set_ylabel('Energy')
ax.set_title('Ground State Energy')
ax.legend()

# Plot 2: ΔE/gap
ax = axes[1]
ax.semilogy(h_train, de_gaps * 100, 'b.-', label='VQE (train)')
ax.scatter([r['h'] for r in deploy_results],
           [r['de_gap'] * 100 for r in deploy_results],
           c='red', s=80, marker='*', zorder=5, label='MPNN (test)')
ax.axhline(5.0, color='gray', ls='--', label='5% threshold')
ax.set_xlabel('h')
ax.set_ylabel('ΔE/gap (%)')
ax.set_title('Relative Energy Error')
ax.legend()

# Plot 3: θ trajectories
ax = axes[2]
for j in range(theta_opt.shape[1]):
    ax.plot(h_train, theta_opt[:, j], '.-', ms=3, label=f'θ_{j}')
ax.set_xlabel('h')
ax.set_ylabel('θ (radians)')
ax.set_title('Optimal Parameters θ*(h)')
ax.legend()

plt.tight_layout()
plt.show()
print("\n🎉 Pipeline complete! The MPNN predicts optimal quantum circuit parameters")
print("   in a single forward pass, eliminating iterative VQE optimization.")